# Main-text figures and tables for the Results chapter


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reported_runs/analysis").is_dir())
sys.path.insert(0, str(repo)) if str(repo) not in sys.path else None
from reported_runs.analysis.presentation import load_results, NotebookReport, label_conditions, rank_summary
from reported_runs.analysis import main_text as mt
%matplotlib inline

results = load_results()
report = NotebookReport(results, 'main')
trajectory = label_conditions(results["trajectory"])
sources = results["sources"]
effect_summaries = results["effect_summaries"]
mt.use_print_style()
main_figures = []
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_rows", 60)

## RQ I: feedback mechanisms


In [ ]:
feedback_effects = effect_summaries.loc[effect_summaries.study.eq('movielens_feedback_a300_e804') & effect_summaries.kind.isin(['feedback', 'individual'])]
F1_LABELS = {'L1 minus L0': 'L1 − L0\n(feature feedback)', 'L2 minus L1': 'L2 − L1\n(model feedback)',
             'L3 full reflection minus L2 frozen agent': 'L3 − L2\n(individual,\nagent only)'}
figure = mt.effect_panels(feedback_effects, ['head_exposure', 'weighted_head_exposure'], categories=list(F1_LABELS),
                          category_labels=F1_LABELS, separators=['L3 full reflection minus L2 frozen agent'], limits=(-25, 25))
main_figures.append(mt.export_main_figure(report, figure, 'main_rq1_feedback_exposure', ""))
report.table(mt.effect_values(figure.plotted, category_labels=F1_LABELS), 'main_rq1_feedback_exposure_values',
             "")

In [ ]:
from reported_runs.analysis.advertising_slots import load_advertising_slot_metrics
from reported_runs.analysis.advertising_views import build_slot_views
sampling_effects = effect_summaries.loc[effect_summaries.kind.eq('sampling')]
slot_sources = sources.loc[sources.study.eq('movielens_sampling_outcome_a150_r150_e804') & sources.condition_id.str.startswith('O')]
slot_metrics = load_advertising_slot_metrics(slot_sources, repo, repo / 'reported_runs/analysis/cache/advertising_slots')
slot_effects = build_slot_views(slot_metrics, results['comparison_plan'])['effect_summaries']
l4_l5 = mt.l4_l5_effect_table(sampling_effects, slot_effects)
report.table(l4_l5, 'main_rq1_l4_l5_effects',
             "", source_frame=pd.concat([sampling_effects, slot_effects]))

In [ ]:
from reported_runs.analysis.sections_rq1_3 import advertising_display
advertising = advertising_display(label_conditions(results['advertising'].loc[
    results['advertising'].study.eq('movielens_sampling_outcome_a150_r150_e804') & results['advertising'].condition_id.str.startswith('O')]))
figure = mt.advertising_delivery(advertising)
main_figures.append(mt.export_main_figure(report, figure, 'main_rq1_advertising_delivery', ""))

## RQ II: user response models


In [ ]:
models_l2 = trajectory.loc[trajectory.study.eq('movielens_feedback_a300_e804') & trajectory.stage.eq('L2_ml_model')]
figure = mt.model_trajectories(models_l2, ['head_click_share', 'click_exposure_gap', 'weighted_click_exposure_gap'],
                               percentage_points=['click_exposure_gap', 'weighted_click_exposure_gap'],
                               limits={'head_click_share': (.4, 1), 'click_exposure_gap': (-20, 20), 'weighted_click_exposure_gap': (-20, 20)})
main_figures.append(mt.export_main_figure(report, figure, 'main_rq2_model_trajectories_l2', ""))

In [ ]:
ranks_l2 = results['ranks'].loc[results['ranks'].study.eq('movielens_feedback_a300_e804') & results['ranks'].stage.eq('L2_ml_model')]
rank_counts = rank_summary(ranks_l2)
figure = mt.rank_panels(rank_counts, arms=('canonical', 'shuffled'))
main_figures.append(mt.export_main_figure(report, figure, 'main_rq2_clicked_ranks_l2', ""))

## RQ III: agent ablations


In [ ]:
from reported_runs.analysis.agent_ablation_views import ABLATION_COMPARISON_ORDER, label_ablation_comparisons
ablations = label_ablation_comparisons(effect_summaries.loc[effect_summaries.study.eq('movielens_feedback_a300_e804') & effect_summaries.kind.eq('ablation')])
F5_LABELS = dict(zip(ABLATION_COMPARISON_ORDER, ['Noise\ncheck', 'Reflection\nnotes', 'Profile\nupdates']))
figure = mt.effect_panels(ablations, ['head_click_share', 'head_exposure', 'weighted_head_exposure'], categories=list(ABLATION_COMPARISON_ORDER),
                          category_labels=F5_LABELS, layout='columns', reference_category=ABLATION_COMPARISON_ORDER[0])
main_figures.append(mt.export_main_figure(report, figure, 'main_rq3_ablation_effects', ""))
report.table(mt.effect_values(figure.plotted, category_labels=F5_LABELS), 'main_rq3_ablation_effects_values',
             "")

In [ ]:
from reported_runs.analysis.agent_usage import load_agent_usage
usage_sources = sources.loc[sources.model.eq('agentic_recommender') & sources.study.eq('movielens_feedback_a300_e804') & sources.stage.str.startswith(('L2', 'L3'))]
usage = load_agent_usage(usage_sources, repo, repo / 'reported_runs/analysis/cache/agent_usage')
agent_diagnostics = label_conditions(results['agent'].loc[results['agent'].study.eq('movielens_feedback_a300_e804')
    & results['agent'].model.eq('agentic_recommender') & results['agent'].stage.str.startswith(('L2', 'L3'))])
module_use = mt.agent_module_table(usage, agent_diagnostics)
report.table(module_use, 'main_rq3_agent_module_use',
             "", source_frame=usage)

## RQ IV: mitigation


In [ ]:
rq4_effects = effect_summaries.loc[effect_summaries.study.eq('movielens_mitigation_a300_e804') & effect_summaries.kind.eq('mitigation')].assign(
    method=lambda frame: frame.intervention.map({'in_processing': 'In-processing', 'post_processing': 'Post-processing'}))
figure = mt.effect_panels(rq4_effects, ['head_click_share', 'head_exposure', 'weighted_head_exposure'], category='method',
                          categories=['In-processing', 'Post-processing'], step=10)
main_figures.append(mt.export_main_figure(report, figure, 'main_rq4_mitigation_effects', ""))
report.table(mt.effect_values(figure.plotted, category='method'), 'main_rq4_mitigation_effects_values',
             "")

In [ ]:
from reported_runs.analysis.mitigation_decomposition import load_postprocessing_components
from reported_runs.analysis.mitigation_decomposition_views import DECOMPOSITION_METRICS, COMPONENT_ORDER, build_postprocessing_decomposition
rq4_sources = sources.loc[sources.study.eq('movielens_mitigation_a300_e804')]
post_components = load_postprocessing_components(rq4_sources.loc[rq4_sources.intervention.eq('post_processing')], repo,
                                                 repo / 'reported_runs/analysis/cache/mitigation_decomposition')
controls = trajectory.loc[trajectory.study.eq('movielens_mitigation_a300_e804') & trajectory.intervention.eq('control')]
component_effects = build_postprocessing_decomposition(post_components, controls, results['comparison_plan'])['summaries']
F7_LABELS = {'Total': 'Total', 'Immediate reranking': 'Immediate\nreranking', 'Feedback component': 'Feedback\ncomponent'}
figure = mt.effect_panels(component_effects, list(DECOMPOSITION_METRICS), category='component_label', categories=list(COMPONENT_ORDER),
                          category_labels=F7_LABELS, step=10)
main_figures.append(mt.export_main_figure(report, figure, 'main_rq4_post_processing_components', ""))
report.table(mt.effect_values(figure.plotted, category='component_label'), 'main_rq4_post_processing_components_values',
             "")

## RQ V: order sensitivity


In [ ]:
order_pairs = results['order_pairs']
figure = mt.order_sensitivity_panels(order_pairs)
main_figures.append(mt.export_main_figure(report, figure, 'main_rq5_order_sensitivity', ""))
order_values = figure.plotted.assign(Model=lambda frame: frame.model.map(mt.MODEL_LABELS))[
    ['predictor_family', 'Model', 'seed', 'baseline_order_sensitivity', 'head_click_reduction_pp']]
report.table(order_values, 'main_rq5_order_sensitivity_values',
             "")

## RQ VI: popularity bias in Goodreads


In [ ]:
from reported_runs.analysis.sections_rq6_7 import POPULARITY, GENDER_ALL, transfer_effects, gender_mitigation_effects
transfer = transfer_effects(effect_summaries)
F9_LABELS = {'L2 minus L0': 'L2 − L0', 'L3 full reflection minus L2 frozen agent': 'L3 − L2\n(agent)'}
figure = mt.effect_panels(transfer, POPULARITY, categories=list(F9_LABELS), category_labels=F9_LABELS, facet='dataset',
                          facet_labels={'movielens': 'MovieLens', 'goodreads': 'Goodreads'},
                          separators=['L3 full reflection minus L2 frozen agent'], limits=(-50, 50))
main_figures.append(mt.export_main_figure(report, figure, 'main_rq6_cross_dataset_popularity', ""))
report.table(mt.effect_values(figure.plotted, category_labels=F9_LABELS, facet='dataset'), 'main_rq6_cross_dataset_popularity_values',
             "")

## RQ VII: author-gender representation in Goodreads


In [ ]:
gender_effects = transfer.loc[transfer.dataset.eq('goodreads')]
figure = mt.effect_panels(gender_effects, GENDER_ALL, categories=list(F9_LABELS), category_labels=F9_LABELS,
                          separators=['L3 full reflection minus L2 frozen agent'])
main_figures.append(mt.export_main_figure(report, figure, 'main_rq7_gender_feedback_effects', ""))
report.table(mt.effect_values(figure.plotted, category_labels=F9_LABELS), 'main_rq7_gender_feedback_effects_values',
             "")

In [ ]:
gender_mitigation = gender_mitigation_effects(effect_summaries)
gender_mitigation_l2 = gender_mitigation.loc[gender_mitigation.stage.eq('L2_ml_model')]
F11_LABELS = {'L2 in-processing minus control': 'In-processing', 'L2 post-processing minus control': 'Post-processing'}
figure = mt.effect_panels(gender_mitigation_l2, GENDER_ALL, categories=list(F11_LABELS), category_labels=F11_LABELS, step=10)
main_figures.append(mt.export_main_figure(report, figure, 'main_rq7_gender_mitigation_l2', ""))
report.table(mt.effect_values(figure.plotted, category_labels=F11_LABELS), 'main_rq7_gender_mitigation_l2_values',
             "")

## Refresh the Goodreads figures in the thesis


In [ ]:
refreshed = ['main_rq6_cross_dataset_popularity', 'main_rq7_gender_feedback_effects']
figure_dir = report.exports / 'figures'
display(Markdown('\n'.join(f'- [{name}.pdf]({(figure_dir / (name + ".pdf")).as_posix()})' for name in refreshed)))